# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:

print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 335.85 GB
MemAvailable: 695.09 GB
Free GPU Memory (GB): 39.3936

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################



In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd

# Define global variables
output_text = ""
is_correct = False
cumulative_prob = None
beams_with_probs = []
top_tokens_with_probs = []

def get_prompt(query, strategy):
    if strategy == "Fact Statement":
        return f"{query} Fact:"
    elif strategy == "Completion":
        return f"{query} The answer is:"
    elif strategy == "Definitive Statement":
        return f"The answer to the question '{query}' is:"
    elif strategy == "True Statement":
        return f"It is true that the answer to '{query}' is:"
    elif strategy == "Declarative Statement":
        return f"{query} The fact is:"
    elif strategy == "Conclusive Statement":
        return f"The final answer to '{query}' is:"
    elif strategy == "Resolved Statement":
        return f"Resolved: '{query}' The answer is:"
    elif strategy == "Ending Completion":
        return f"{query} The final answer is:"
    elif strategy == "Answer Completion":
        return f"{query} The correct answer is:"
    elif strategy == "Plain Completion":
        return f"{query} The answer:"
    elif strategy == "Direct Completion":
        return f"{query} Answer:"
    elif strategy == "Simple Completion":
        return f"{query} Result:"
    elif strategy == "Direct Answer":
        return f"{query} Correct answer:"
    elif strategy == "Answer Statement":
        return f"{query} The exact answer is:"
    elif strategy == "True Completion":
        return f"{query} The true answer is:"
    else:
        return query

def clean_response(output_text, strategy):
    if strategy == "Fact Statement":
        return output_text.split("Fact:")[-1].strip()
    elif strategy == "Completion":
        return output_text.split("The answer is:")[-1].strip()
    elif strategy == "Definitive Statement":
        return output_text.split("is:")[-1].strip()
    elif strategy == "True Statement":
        return output_text.split("is:")[-1].strip()
    elif strategy == "Declarative Statement":
        return output_text.split("The fact is:")[-1].strip()
    elif strategy == "Conclusive Statement":
        return output_text.split("is:")[-1].strip()
    elif strategy == "Resolved Statement":
        return output_text.split("is:")[-1].strip()
    elif strategy == "Ending Completion":
        return output_text.split("is:")[-1].strip()
    elif strategy == "Answer Completion":
        return output_text.split("The correct answer is:")[-1].strip()
    elif strategy == "Plain Completion":
        return output_text.split("The answer:")[-1].strip()
    elif strategy == "Direct Completion":
        return output_text.split("Answer:")[-1].strip()
    elif strategy == "Simple Completion":
        return output_text.split("Result:")[-1].strip()
    elif strategy == "Direct Answer":
        return output_text.split("Correct answer:")[-1].strip()
    elif strategy == "Answer Statement":
        return output_text.split("The exact answer is:")[-1].strip()
    elif strategy == "True Completion":
        return output_text.split("The true answer is:")[-1].strip()
    return output_text

model_name = "meta-llama/Meta-Llama-3-8B"
query = "What is your favorite colour?"
true_answer = "Paris"
max_new_tokens = 20
temperature = 1
strategy = "Definitive Statement"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
model.eval()

prompt = get_prompt(query, strategy)
input_ids = tokenizer.encode(prompt, return_tensors='pt').to("cuda")
generation_config = {
    "temperature": temperature,
    "do_sample": True,
    "top_p": 0.75,
    "top_k": 40,
    "output_scores": True,
    "output_hidden_states": False,
    "output_attentions": False,
    "num_beams": 5,
    "num_return_sequences": 5,
    "return_dict_in_generate": True,
    "stop_strings": ["."]
}

output_texts = []
output_list = []
for i in range(20):
    with torch.no_grad():
        outputs = model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens, tokenizer=tokenizer)

    output_text = tokenizer.decode(outputs[0][0], skip_special_tokens=True)
    output_text = clean_response(output_text, strategy)
    response_tokens = outputs[0].tolist()
    output_scores = outputs.scores
    output_texts.append(output_text)
    output_list.append(outputs)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.29it/s]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable r

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd
import numpy as np

# Define global variables
n_answers_per_question = 5
questions = [
    "What is your favorite colour?",
]
output_texts = []
common_prefixes = []
cumulative_probs = []
adjusted_probs = []
entropy_scores = []
tokens_list = []
token_probs_list = []
prompted_questions = []
full_output_texts = []
output_list

def get_prompt(query, strategy):
    if strategy == "Fact Statement":
        return f"{query} Fact:"
    elif strategy == "Completion":
        return f"{query} The answer is:"
    elif strategy == "Definitive Statement":
        return f"The answer to the question '{query}' is:"
    # Add other strategies if needed
    return query

def clean_response(output_text, strategy):
    if strategy == "Fact Statement":
        return output_text.split("Fact:")[-1].strip()
    elif strategy == "Completion":
        return output_text.split("The answer is:")[-1].strip()
    elif strategy == "Definitive Statement":
        return output_text.split("is:")[-1].strip()
    # Add other strategies if needed
    return output_text

def calculate_entropy(probs):
    return -np.sum(probs * np.log(probs))

# Initialize model and tokenizer
model_name = "meta-llama/Meta-Llama-3-8B"
max_new_tokens = 20
temperature = 1
strategy = "Definitive Statement"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
model.eval()

generation_config = {
    "temperature": temperature,
    "do_sample": True,
    "top_p": 0.75,
    "top_k": 40,
    "output_scores": True,
    "output_hidden_states": False,
    "output_attentions": False,
    # "num_beams": 5,
    # "num_return_sequences": 1,
    "return_dict_in_generate": True,
    "stop_strings": ["."]
}

# test the idea if we get the same outputs for num_beams vs no num_beams

for query in questions:
    prompt = get_prompt(query, strategy)
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to("cuda")
    
    for _ in range(n_answers_per_question):
        with torch.no_grad():
            outputs = model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens, tokenizer=tokenizer)

        full_output_text = tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        clean_output_text = clean_response(full_output_text, strategy)
        
        # Determine the common prefix
        prefix_length = len(full_output_text) - len(clean_output_text)
        common_prefix = full_output_text[:prefix_length]
        
        # Store the common prefix and the cleaned response
        common_prefixes.append(common_prefix)
        output_texts.append(clean_output_text)
        full_output_texts.append(full_output_text)
        prompted_questions.append(prompt)
        
        # Calculate probabilities for the generated part
        probs = []
        token_probs = []
        sequence = outputs.sequences[0]
        cum_prob = 1.0
        
        shift_idx = len(sequence) - len(outputs.scores)  # Position to start processing tokens
        for pos_idx, beam_scores in enumerate(outputs.scores):
            softmax_scores = torch.softmax(beam_scores, dim=-1)
            token_prob = softmax_scores[0][sequence[pos_idx + shift_idx]].item()
            token_probs.append(round(token_prob, 4))
            cum_prob *= token_prob
            probs.append(token_prob)
        
        probs = torch.tensor(probs)
        adj_prob = torch.exp((len(probs) ** -1) * torch.sum(torch.log(probs)))
        
        # Calculate entropy score
        entropy = calculate_entropy(np.array(probs))
        
        # Append results
        cumulative_probs.append(round(cum_prob, 4))
        adjusted_probs.append(round(adj_prob.item(), 4))
        entropy_scores.append(round(entropy, 4))
        tokens_list.append([tokenizer.decode([token]) for token in sequence[shift_idx:]])
        token_probs_list.append(token_probs)

# Create DataFrame
df = pd.DataFrame({
    "Original Question": np.repeat(questions, n_answers_per_question),
    # "Full Output Text": full_output_texts,
    "Prompted Question": prompted_questions,
    # "Common Prefix": common_prefixes,
    "Generated Answer": output_texts,
    "Cumulative Probability": cumulative_probs,
    "Adjusted Probability": adjusted_probs,
    "Entropy Score": entropy_scores,
    "Tokens": tokens_list,
    "Token Probabilities": token_probs_list
})

# Save to Excel
df.to_excel("multinomial_sampling_results-one-prompt.xlsx", index=False)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.23it/s]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/tmp/ipykernel_338016/3327513856.py:42: RuntimeWarning: divide by zero encountered in log
  return -np.sum(probs * np.log(probs))
/tmp/ipykernel_338016/3327513856.py:42: RuntimeWarning: invalid value encountered in multiply
  return -np.sum(probs * np.log(probs))
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Sett

In [16]:
outputs = output_list[1]

In [17]:
tokenizer.decode(outputs.sequences[0])

"<|begin_of_text|>The answer to the question 'What is your favorite colour?' is: 'I don't have a favourite colour.'\nThe answer to the question 'What is your favourite colour"

In [18]:
len(outputs.scores)

20

In [20]:
sequence = outputs.sequences[0]

In [28]:
for token in sequence[10:]:
    print(token)
    print(tokenizer.decode([token]))

tensor(6864, device='cuda:0')
 capital
tensor(3363, device='cuda:0')
 city
tensor(315, device='cuda:0')
 of
tensor(9822, device='cuda:0')
 France
tensor(20837, device='cuda:0')
?'
tensor(374, device='cuda:0')
 is
tensor(25, device='cuda:0')
:
tensor(12366, device='cuda:0')
 Paris
tensor(627, device='cuda:0')
.

tensor(791, device='cuda:0')
The
tensor(6864, device='cuda:0')
 capital
tensor(3363, device='cuda:0')
 city
tensor(315, device='cuda:0')
 of
tensor(9822, device='cuda:0')
 France
tensor(374, device='cuda:0')
 is
tensor(12366, device='cuda:0')
 Paris
tensor(627, device='cuda:0')
.

tensor(791, device='cuda:0')
The
tensor(6864, device='cuda:0')
 capital
tensor(3363, device='cuda:0')
 city
tensor(315, device='cuda:0')
 of
tensor(9822, device='cuda:0')
 France
tensor(374, device='cuda:0')
 is
tensor(12366, device='cuda:0')
 Paris
tensor(627, device='cuda:0')
.

tensor(791, device='cuda:0')
The
tensor(6864, device='cuda:0')
 capital


In [23]:
tokenizer.decode([791])

'Paris'

In [22]:
tokenizer.encode(output_text)

[128000,
 60704,
 627,
 791,
 6864,
 3363,
 315,
 9822,
 374,
 12366,
 627,
 791,
 6864,
 3363,
 315,
 9822,
 374,
 12366,
 627,
 791,
 6864]

In [14]:
sequence = outputs.sequences[0]
sequence[20:]

tensor([   25,   364,  1163,   391,  1138,   278, 36231,   596, 13166, 26097,
          374,  7559,   304, 10016, 84782,    11, 17368,  3238],
       device='cuda:0')

In [10]:
tokenizer.encode(output_text)

[128000,
 6,
 1163,
 391,
 1138,
 278,
 36231,
 596,
 13166,
 26097,
 374,
 7559,
 304,
 10016,
 84782,
 11,
 17368,
 3238]

In [51]:
tokenizer.decode(output_list[0].sequences[0][14:])

': green.'

In [90]:
print(output_list[0].sequences[0][14:])
print(tokenizer.decode(output_list[0].sequences[0][14:]))

tensor([  25, 6307,   13], device='cuda:0')
: green.


In [96]:
len(output_list[0].scores)

2

In [91]:
torch.softmax(output_list[0].scores[0], dim=-1)[0][6307]

tensor(0.0313, device='cuda:0')

In [99]:
len(output_list[0].sequences[0])

17

In [101]:
len(output_list[0].scores)

2

In [109]:
import numpy as np

for output_idx, output in enumerate(output_list[:3]):
  cum_prob = 1.0
  probs = []
  print(f"Output {output_idx}:")
  sequence = output.sequences[0]
  beam_tokens = np.zeros((5, 20), dtype=int)
  for pos_idx, beam_scores in enumerate(output.scores):
    softmax_scores = torch.softmax(beam_scores, dim=-1)
    shift_idx = len(sequence) - len(output.scores)
    prob = softmax_scores[0][sequence[pos_idx + shift_idx]].item()
    cum_prob *= prob
    probs.append(prob)
    print(f"  Position {pos_idx}, Token: {tokenizer.decode([sequence[pos_idx + 15]])} (P={prob:.4f})")
      # for token_idx, token in enumerate(token_list):
        # print(f"    {token_idx}: '{tokenizer.decode([token])}'", end=" ")
  print(f"  Cumulative Probability: {cum_prob:.4f}")
  probs = torch.tensor(probs)
  adj_prob = torch.exp((len(probs) ** -1) * torch.sum(torch.log(probs)))
  print(f"  Adjusted Probability: {adj_prob:.4f}")

Output 0:
  Position 0, Token:  green (P=0.0313)
  Position 1, Token: . (P=0.4469)
  Cumulative Probability: 0.0140
  Adjusted Probability: 0.1183
Output 1:
  Position 0, Token:  I (P=0.0623)
  Position 1, Token:  don (P=0.4614)
  Position 2, Token: 't (P=1.0000)
  Position 3, Token:  have (P=0.7982)
  Position 4, Token:  a (P=0.4073)
  Position 5, Token:  favourite (P=0.5000)
  Position 6, Token:  colour (P=1.0000)
  Position 7, Token: .
 (P=0.3262)
  Cumulative Probability: 0.0015
  Adjusted Probability: 0.4445
Output 2:
  Position 0, Token:  ' (P=0.4325)
  Position 1, Token: Blue (P=0.1312)
  Position 2, Token: ' (P=0.1255)
  Position 3, Token: !
 (P=0.1343)
  Position 4, Token: I (P=0.1948)
  Position 5, Token:  am (P=0.1663)
  Position 6, Token:  a (P=0.4377)
  Position 7, Token:   (P=0.1547)
  Position 8, Token: 24 (P=0.0534)
  Position 9, Token:  year (P=0.6514)
  Position 10, Token:  old (P=1.0000)
  Position 11, Token:  male (P=0.0929)
  Position 12, Token:  from (P=0.2461)
  

In [18]:
len(outputs.sequences[0]) - len(outputs.scores)

15

In [51]:
softmax_scores.topk(k=200, dim=-1).indices.tolist()

[[20837,
  30,
  0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
  

In [50]:
softmax_scores

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')

In [56]:
len(outputs.scores)

20

In [24]:
beam_scores

tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf],
        [-inf, -inf, -inf,  ..., -inf, -inf, -inf],
        [-inf, -inf, -inf,  ..., -inf, -inf, -inf],
        [-inf, -inf, -inf,  ..., -inf, -inf, -inf],
        [-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')

In [25]:
torch.nn.functional.log_softmax(beam_scores, dim=-1)

tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf],
        [-inf, -inf, -inf,  ..., -inf, -inf, -inf],
        [-inf, -inf, -inf,  ..., -inf, -inf, -inf],
        [-inf, -inf, -inf,  ..., -inf, -inf, -inf],
        [-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0')

In [27]:
torch.exp(torch.nn.functional.log_softmax(beam_scores, dim=-1)).topk(k=3, dim=-1)

torch.return_types.topk(
values=tensor([[0.9099, 0.0901, 0.0000],
        [0.8741, 0.1259, 0.0000],
        [0.6109, 0.1281, 0.0644],
        [0.8808, 0.1192, 0.0000],
        [0.2750, 0.1890, 0.1775]], device='cuda:0'),
indices=tensor([[12745,  3691,     0],
        [12745,  1933,     0],
        [12745,  3691,  4632],
        [19214,  7075,     0],
        [26159, 20820, 19214]], device='cuda:0'))

In [29]:
torch.softmax(beam_scores, dim=-1).topk(k=3, dim=-1)

torch.return_types.topk(
values=tensor([[0.9099, 0.0901, 0.0000],
        [0.8741, 0.1259, 0.0000],
        [0.6109, 0.1281, 0.0644],
        [0.8808, 0.1192, 0.0000],
        [0.2750, 0.1890, 0.1775]], device='cuda:0'),
indices=tensor([[12745,  3691,     0],
        [12745,  1933,     0],
        [12745,  3691,  4632],
        [19214,  7075,     0],
        [26159, 20820, 19214]], device='cuda:0'))

In [31]:
import numpy as np
import torch

# Assuming cum_probs and beam_tokens are initialized with appropriate dimensions
cum_probs = np.ones((5, 20))
beam_tokens = np.zeros((5, 20), dtype=int)

# Iterate over each position index and the corresponding beam scores
for pos_idx, beam_scores in enumerate(outputs.scores):
    print(f"Position {pos_idx}:")

    # Calculate the softmax scores for the current position
    softmax_scores = torch.exp(torch.nn.functional.log_softmax(beam_scores, dim=-1))

    # Iterate over each beam
    for beam_idx in range(beam_scores.size(0)):
        # Get the true token from the generated sequence
        true_token_index = pos_idx + c
        true_token = outputs.sequences[beam_idx][true_token_index]

        # Decode the true token for display
        decoded_token = tokenizer.decode([true_token])
        print(f"  Beam {beam_idx}, True Token: {decoded_token}")

        # Get the probability of the true token from the softmax scores
        prob = softmax_scores[beam_idx, true_token].item()

        # Store the true token and its probability
        beam_tokens[beam_idx, pos_idx] = true_token
        cum_probs[beam_idx, pos_idx] = prob

        # Display the token and its probability
        print(f"    Probability: P({decoded_token}) = {prob}")

    print("")

# Print the cumulative probabilities
print(f"Cumulative Probabilities: {cum_probs}")


Position 0:
  Beam 0, True Token:  '
    Probability: P( ') = 0.4377550184726715
  Beam 1, True Token:  '
    Probability: P( ') = 0.4377550184726715
  Beam 2, True Token:  '
    Probability: P( ') = 0.4377550184726715
  Beam 3, True Token:  '
    Probability: P( ') = 0.4377550184726715
  Beam 4, True Token:  '
    Probability: P( ') = 0.4377550184726715

Position 1:
  Beam 0, True Token: I
    Probability: P(I) = 0.18789616227149963
  Beam 1, True Token: I
    Probability: P(I) = 0.2523956000804901
  Beam 2, True Token: I
    Probability: P(I) = 0.0
  Beam 3, True Token: I
    Probability: P(I) = 0.0
  Beam 4, True Token: I
    Probability: P(I) = 0.0

Position 2:
  Beam 0, True Token:  don
    Probability: P( don) = 0.0
  Beam 1, True Token:  don
    Probability: P( don) = 0.5527547001838684
  Beam 2, True Token:  don
    Probability: P( don) = 0.0
  Beam 3, True Token:  don
    Probability: P( don) = 0.0
  Beam 4, True Token:  don
    Probability: P( don) = 0.0

Position 3:
  Beam 0

In [23]:
np.exp(transition_scores.cpu().numpy())

array([[0.437755  , 0.18789613, 0.5527547 , 0.9579122 , 0.79818684,
        0.5312094 , 0.5       , 0.9149009 , 0.13012274, 0.29948038,
        0.8207173 , 0.97404265, 0.9859364 , 0.9978172 , 0.9914224 ,
        0.9324533 , 0.9525741 , 0.97404265, 0.56217647, 0.909907  ],
       [0.437755  , 0.18789613, 0.5527547 , 0.9579122 , 0.79818684,
        0.5312094 , 0.5       , 0.9149009 , 0.13012274, 0.29948038,
        0.8207173 , 0.97404265, 0.9859364 , 0.9978172 , 0.9914224 ,
        0.9324533 , 0.9525741 , 0.97404265, 0.43782347, 0.87407726],
       [0.437755  , 0.18789613, 0.5527547 , 0.9579122 , 0.79818684,
        0.5312094 , 0.5       , 0.9149009 , 0.13012274, 0.29948038,
        0.8207173 , 0.97404265, 0.9859364 , 0.9978172 , 0.9914224 ,
        0.9324533 , 0.9525741 , 0.97404265, 0.43782347, 0.12592277],
       [0.437755  , 0.18789613, 0.5527547 , 0.9579122 , 0.79818684,
        0.5312094 , 0.5       , 0.9149009 , 0.13012274, 0.29948038,
        0.8207173 , 0.97404265, 0.9859364 , 0

In [21]:
transition_scores = model.compute_transition_scores(
  outputs.sequences,
  outputs.scores,
  normalize_logits=True,
  beam_indices=outputs.beam_indices
)
for i in range(len(transition_scores)):
    prob=np.exp(transition_scores.cpu().numpy())[i].prod(axis=0)
    print(prob/len(transition_scores[i]))

5.6795866839820516e-06
4.249090125085786e-06
6.121395017544273e-07
5.623552169709e-07
2.933896212198306e-07


In [22]:
np.exp(transition_scores.cpu().numpy())

array([[0.437755  , 0.18789613, 0.5527547 , 0.9579122 , 0.79818684,
        0.5312094 , 0.5       , 0.9149009 , 0.13012274, 0.29948038,
        0.8207173 , 0.97404265, 0.9859364 , 0.9978172 , 0.9914224 ,
        0.9324533 , 0.9525741 , 0.97404265, 0.56217647, 0.909907  ],
       [0.437755  , 0.18789613, 0.5527547 , 0.9579122 , 0.79818684,
        0.5312094 , 0.5       , 0.9149009 , 0.13012274, 0.29948038,
        0.8207173 , 0.97404265, 0.9859364 , 0.9978172 , 0.9914224 ,
        0.9324533 , 0.9525741 , 0.97404265, 0.43782347, 0.87407726],
       [0.437755  , 0.18789613, 0.5527547 , 0.9579122 , 0.79818684,
        0.5312094 , 0.5       , 0.9149009 , 0.13012274, 0.29948038,
        0.8207173 , 0.97404265, 0.9859364 , 0.9978172 , 0.9914224 ,
        0.9324533 , 0.9525741 , 0.97404265, 0.43782347, 0.12592277],
       [0.437755  , 0.18789613, 0.5527547 , 0.9579122 , 0.79818684,
        0.5312094 , 0.5       , 0.9149009 , 0.13012274, 0.29948038,
        0.8207173 , 0.97404265, 0.9859364 , 0

In [38]:
cum_prob_list = cum_probs.prod(axis=1)
cum_prob_list

array([0.0016833 , 0.00024142, 0.00068468, 0.00051418, 0.00859551])

In [ ]:
last_idx_cum_prob = 

In [29]:
print([len(sequence) for sequence in outputs.sequences])

[35, 35, 35, 35, 35]


In [34]:
outputs.sequences_scores

tensor([-0.7514, -0.7572, -0.7808, -0.7910, -0.7943], device='cuda:0')

In [25]:
torch.exp(outputs.sequences_scores[0] * len(outputs.sequences[0]))

tensor(3.7905e-12, device='cuda:0')

In [72]:
output_texts

['green.',
 "I don't have a favourite colour.",
 "'Blue'!\nI am a 24 year old male from England.",
 'Red, because red is the colour of my favorite fruit: a strawberry.',
 "'I have no favourite colour.'",
 '"I have a lot of favorite colours."',
 'Blue.',
 "'The one that goes best with the rest of my outfit'.",
 "'It depends on the context and my mood at the time.'",
 "'Green, blue, red, black, white, yellow, pink, purple, brown, orange",
 "'All of them'.",
 'Black.',
 'Red.',
 '0.',
 'pink.',
 "'It's not the same for everyone.'",
 'Blue.',
 "'It depends on the light' - or so the latest research on colour perception suggests.",
 "'I don't have a favourite colour.",
 "'Blue.'"]

In [73]:
tokenizer.decode(output_list[0].sequences[0][14:])

': green.'

In [69]:
softmax_scores = torch.softmax(output_list[0].scores[0], dim=-1).topk(k=5, dim=-1).indices.tolist()
tokenizer.decode(softmax_scores[0])

'\'" blue I Blue'

In [58]:
tokenizer.decode(sequence)

"<|begin_of_text|>The answer to the question 'What is your favorite colour?' is: green."

In [59]:
tokenizer.decode(token_list)

"'.\n'.'.',"

In [20]:
softmax_scores = torch.softmax(output_list[0].scores[16], dim=-1)
softmax_tokens = softmax_scores.topk(k=3, dim=-1).indices.tolist()
for token in softmax_tokens:
    print(token)

[24314, 3238, 11]


In [23]:
tokenizer.decode([11])

','

In [27]:
import torch
torch.exp(outputs.sequences_scores)

tensor([0.0270, 0.0196, 0.0154, 0.0149, 0.0135, 0.0118, 0.0106, 0.0087, 0.0081,
        0.0071, 0.0060, 0.0058, 0.0055, 0.0044, 0.0039, 0.0038, 0.0036, 0.0036,
        0.0035, 0.0033, 0.0032, 0.0031, 0.0031, 0.0029, 0.0027, 0.0026, 0.0026,
        0.0025, 0.0025, 0.0024], device='cuda:0')

In [23]:
len(outputs.sequences[0]) - 1

36

In [16]:
torch.exp(outputs.sequences_scores) * (len(outputs.sequences) - 1)

tensor([0.2275, 0.1728, 0.1469, 0.1432, 0.1298, 0.1100, 0.0875, 0.0695, 0.0684,
        0.0524], device='cuda:0')

In [25]:
sum(torch.exp(outputs.sequences_scores))

tensor(0.2078, device='cuda:0')

In [9]:
tokenizer.decode(outputs.sequences[0])

"<|begin_of_text|>The answer to the question 'What is the capital city of France?' is: Paris.\nThe capital city of France is Paris.\nThe capital city of France is Paris.\nThe capital"

In [14]:
torch.exp(outputs.sequences_scores[0].cpu())

tensor(0.0047)

In [10]:
output_text

'Paris.\nThe capital city of France is Paris.\nThe capital city of France is Paris.\nThe capital'

In [4]:
import numpy as np

beam_tokens = np.zeros((5, 20), dtype=int)
for pos_idx, beam_scores in enumerate(output_scores):
  print(f"Position {pos_idx}:")
  softmax_scores = torch.softmax(beam_scores, dim=-1)
  argmax_tokens = softmax_scores.topk(k=3, dim=-1).indices.tolist()
  for beam_idx, token_list in enumerate(argmax_tokens):
    print(f"  Beam {beam_idx}:")
    beam_tokens[beam_idx, pos_idx] = token_list[0]
    for token_idx, token in enumerate(token_list):
      print(f"    {token_idx}: '{tokenizer.decode([token])}'", end=" ")
    print("")

Position 0:
  Beam 0:
    0: ' Paris'     1: ' ''     2: '!' 
  Beam 1:
    0: ' Paris'     1: ' ''     2: '!' 
  Beam 2:
    0: ' Paris'     1: ' ''     2: '!' 
  Beam 3:
    0: ' Paris'     1: ' ''     2: '!' 
  Beam 4:
    0: ' Paris'     1: ' ''     2: '!' 
Position 1:
  Beam 0:
    0: '.'     1: '.
'     2: '!' 
  Beam 1:
    0: 'Paris'     1: 'The'     2: '!' 
  Beam 2:
    0: ' Paris'     1: ' The'     2: '!' 
  Beam 3:
    0: 'Paris'     1: ' Paris'     2: '!' 
  Beam 4:
    0: ' Paris'     1: 'Paris'     2: '!' 
Position 2:
  Beam 0:
    0: ' The'     1: ' Paris'     2: '!' 
  Beam 1:
    0: 'The'     1: 'Paris'     2: '!' 
  Beam 2:
    0: ''.'     1: ''.
'     2: '!' 
  Beam 3:
    0: ' capital'     1: ' answer'     2: '!' 
  Beam 4:
    0: 'period'     1: 'comma'     2: '!' 
Position 3:
  Beam 0:
    0: ' The'     1: ' To'     2: '!' 
  Beam 1:
    0: ' capital'     1: ' answer'     2: '!' 
  Beam 2:
    0: ' capital'     1: ' answer'     2: '!' 
  Beam 3:
    0: ' is'     

In [63]:
softmax_scores = torch.softmax(output_scores[6], dim=-1)
softmax_tokens = softmax_scores.topk(k=3, dim=-1).indices.tolist()
decoded_tokens = [tokenizer.decode(token) for token in softmax_tokens]
softmax_tokens

[[13, 627, 0],
 [60704, 791, 0],
 [12366, 578, 0],
 [60704, 12366, 0],
 [12366, 60704, 0]]

In [67]:
softmax_scores.topk(k=3, dim=-1)

torch.return_types.topk(
values=tensor([[9.2414e-01, 7.5858e-02, 0.0000e+00],
        [1.0000e+00, 1.1400e-12, 0.0000e+00],
        [9.9331e-01, 6.6928e-03, 0.0000e+00],
        [1.0000e+00, 5.9053e-10, 0.0000e+00],
        [9.9984e-01, 1.5844e-04, 0.0000e+00]], device='cuda:0'),
indices=tensor([[   13,   627,     0],
        [60704,   791,     0],
        [12366,   578,     0],
        [60704, 12366,     0],
        [12366, 60704,     0]], device='cuda:0'))

In [49]:
beam_tokens.shape
for beam_idx,  beam in enumerate(beam_tokens):
  print(f"Beam {beam_idx}: {tokenizer.decode(beam)}")

Beam 0:  Paris. The The the capital city is Paris.
The capital located capital city is Paris.<|end_of_text|> capital
Beam 1:  ParisParisThe capital city of France Paris France. It is:// of France of France.
Paris<|begin_of_text|>
Beam 2:  Paris Paris'. capital of France is of..<|end_of_text|> is theen the France. andThe capital
Beam 3:  ParisParis capital is capital France is Paris Paris Paris The city city in city north of It<|end_of_text|> is
Beam 4:  Paris ParisperiodThe learn of France is. TheThe<|begin_of_text|> is largest.wikipedia in France is the<|begin_of_text|>


In [54]:
output_text

'Paris.\nThe capital city of France is Paris.\nThe capital city of France is Paris.'

In [52]:
tokenizer.encode(output_text, return_tensors='pt')

tensor([[128000,  60704,    627,    791,   6864,   3363,    315,   9822,    374,
          12366,    627,    791,   6864,   3363,    315,   9822,    374,  12366,
             13]])

In [68]:
tokenizer.decode([13])

'.'

In [6]:
df.groupby(["model", "strategy", "max_new_tokens", "temp"]).agg({
    "is_correct": ["sum"],
    "cum_prob": ["mean"]
}).to_excel("results_07_31_v2_agg.xlsx")

In [48]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd

# Define global variables
output_text = ""
is_correct = False
cumulative_prob = None
beams_with_probs = []
top_tokens_with_probs = []

def get_prompt(query, strategy):
    if strategy == "Fact Statement":
        return f"{query} Fact:"
    elif strategy == "Completion":
        return f"{query} The answer is:"
    elif strategy == "Definitive Statement":
        return f"The answer to the question '{query}' is:"
    elif strategy == "True Statement":
        return f"It is true that the answer to '{query}' is:"
    elif strategy == "Declarative Statement":
        return f"{query} The fact is:"
    elif strategy == "Conclusive Statement":
        return f"The final answer to '{query}' is:"
    elif strategy == "Resolved Statement":
        return f"Resolved: '{query}' The answer is:"
    elif strategy == "Ending Completion":
        return f"{query} The final answer is:"
    elif strategy == "Answer Completion":
        return f"{query} The correct answer is:"
    elif strategy == "Plain Completion":
        return f"{query} The answer:"
    elif strategy == "Direct Completion":
        return f"{query} Answer:"
    elif strategy == "Simple Completion":
        return f"{query} Result:"
    elif strategy == "Direct Answer":
        return f"{query} Correct answer:"
    elif strategy == "Answer Statement":
        return f"{query} The exact answer is:"
    elif strategy == "True Completion":
        return f"{query} The true answer is:"
    else:
        return query

def clean_response(output_text, strategy):
    if strategy == "Fact Statement":
        return output_text.split("Fact:")[-1].strip()
    elif strategy == "Completion":
        return output_text.split("The answer is:")[-1].strip()
    elif strategy == "Definitive Statement":
        return output_text.split("is:")[-1].strip()
    elif strategy == "True Statement":
        return output_text.split("is:")[-1].strip()
    elif strategy == "Declarative Statement":
        return output_text.split("The fact is:")[-1].strip()
    elif strategy == "Conclusive Statement":
        return output_text.split("is:")[-1].strip()
    elif strategy == "Resolved Statement":
        return output_text.split("is:")[-1].strip()
    elif strategy == "Ending Completion":
        return output_text.split("is:")[-1].strip()
    elif strategy == "Answer Completion":
        return output_text.split("The correct answer is:")[-1].strip()
    elif strategy == "Plain Completion":
        return output_text.split("The answer:")[-1].strip()
    elif strategy == "Direct Completion":
        return output_text.split("Answer:")[-1].strip()
    elif strategy == "Simple Completion":
        return output_text.split("Result:")[-1].strip()
    elif strategy == "Direct Answer":
        return output_text.split("Correct answer:")[-1].strip()
    elif strategy == "Answer Statement":
        return output_text.split("The exact answer is:")[-1].strip()
    elif strategy == "True Completion":
        return output_text.split("The true answer is:")[-1].strip()
    return output_text

model_name = "meta-llama/Meta-Llama-3-8B"
query = "Where is Simcoe composite school located?"
true_answer = "Paris"
max_new_tokens = 20
temperature = 0.1
strategy = "Definitive Statement"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
model.eval()

prompt = get_prompt(query, strategy)
input_ids = tokenizer.encode(prompt, return_tensors='pt').to("cuda")
generation_config = {
    "temperature": temperature,
    "do_sample": True,
    "top_p": 0.75,
    "top_k": 40,
    "num_beams": 5,
    "num_return_sequences": 5,
    "output_scores": True,
    "output_hidden_states": False,
    "output_attentions": False,
    "return_dict_in_generate": True
}

with torch.no_grad():
    outputs = model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens)

output_text = tokenizer.decode(outputs[0][0], skip_special_tokens=True)
output_text = clean_response(output_text, strategy)
response_tokens = outputs[0].tolist()
output_scores = outputs.scores

def check_answer(tokenizer, response_tokens, true_answer, output_scores, input_ids):
    generated_tokens = response_tokens[input_ids.size(1):]
    if len(generated_tokens) == 0:
        return False, None, None, None
    assert len(generated_tokens) == len(output_scores)
    
    # Generate the four variants of true_answer
    true_answer_lower = true_answer.lower()
    true_answer_title = true_answer.title()
    true_tokens_no_space_lower = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(true_answer_lower))
    true_tokens_no_space_title = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(true_answer_title))
    true_tokens_with_space_lower = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(" " + true_answer_lower))
    true_tokens_with_space_title = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(" " + true_answer_title))

    variants = [
        true_tokens_no_space_lower,
        true_tokens_no_space_title,
        true_tokens_with_space_lower,
        true_tokens_with_space_title
    ]

    def get_cumulative_probability(true_tokens, idx, output_scores):
        cumulative_prob = 1.0
        top_tokens_with_probs = []
        beams_with_probs = []

        for true_token_idx, true_token in enumerate(true_tokens):
            for beam_index, beam_scores in enumerate(output_scores[idx + true_token_idx]):
                token_probs = torch.softmax(beam_scores, dim=-1)
                token_prob = token_probs[true_token].item()
                top_indices = (token_probs >= 0.1).nonzero(as_tuple=True)[0]
                top_probs = token_probs[top_indices]
                top_tokens = tokenizer.convert_ids_to_tokens(top_indices)
                top_tokens = [token.replace("Ġ", " ") for token in top_tokens]

                if true_token in top_indices:
                    top_tokens_with_probs.extend([(token, prob.item()) for token, prob in zip(top_tokens, top_probs)])
                    cumulative_prob *= token_prob
                    beams_with_probs.append({
                        'character_index': idx + true_token_idx,
                        'beam_index': beam_index,
                        'token': tokenizer.decode([true_token]),
                        'token_index': true_token,
                        'probability': token_prob
                    })
                    break

        return cumulative_prob, beams_with_probs, top_tokens_with_probs

    for variant in variants:
        for idx in range(len(generated_tokens) - len(variant) + 1):
            if generated_tokens[idx:idx + len(variant)] == variant:
                cumulative_prob, beams_with_probs, top_tokens_with_probs = get_cumulative_probability(variant, idx, output_scores)
                return True, cumulative_prob, beams_with_probs, top_tokens_with_probs

    return False, None, None, None

is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs = check_answer(tokenizer, response_tokens[0], true_answer, output_scores, input_ids)

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.09s/it]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [43]:
tokenizer.decode(outputs.sequences[0])

"<|begin_of_text|>The answer to the question 'e sges  reseee s se as' is: 'e sges  reseee s se as' is 'e sges  rese"

(tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:0'),
 tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cu

In [51]:
sequence = outputs.sequences[0]
tokenizer.decode(sequence[17:])

' Simcoe Composite School is located at 1000 Simcoe Street, Simcoe, Ontario, Canada'

In [50]:
output_text

'Simcoe Composite School is located at 1000 Simcoe Street, Simcoe, Ontario, Canada'

In [36]:
tokenizer.encode(output_text)

[128000,
 17773,
 383,
 6864,
 3363,
 315,
 6907,
 71,
 5981,
 383,
 374,
 6907,
 71,
 5981,
 383,
 24314,
 791,
 4320,
 311,
 279,
 3488]

In [35]:
import numpy as np

sequence = outputs.sequences[0]
cum_prob = 1.0
beam_tokens = np.zeros((5, 20), dtype=int)
for pos_idx, beam_scores in enumerate(output_scores):
  print(f"Position {pos_idx}:")
  softmax_scores = torch.softmax(beam_scores, dim=-1)
  argmax_tokens = softmax_scores.topk(k=5, dim=-1).indices.tolist()
  for beam_idx, token_list in enumerate(argmax_tokens):
    # print(f"  Beam {beam_idx}:")
    beam_tokens[beam_idx, pos_idx] = token_list[0]
    for token_idx, token in enumerate(token_list):
      if token == sequence[pos_idx + 17]:
        prob = softmax_scores[beam_idx][token].item()
        cum_prob *= prob
        print(f"  Beam {beam_idx}, Token {token_idx}: {tokenizer.decode([token])} (P={prob})")
    # for token_idx, token in enumerate(token_list):
      # print(f"    {token_idx}: '{tokenizer.decode([token])}'", end=" ")
    print("")

Position 0:
  Beam 0, Token 0:  Paris (P=1.0)

Position 1:
  Beam 0, Token 0: . (P=1.0)

Position 2:
  Beam 0, Token 0:  The (P=1.0)

Position 3:
  Beam 0, Token 0:  capital (P=1.0)

Position 4:
  Beam 0, Token 0:  of (P=1.0)

Position 5:
  Beam 0, Token 0:  France (P=1.0)

Position 6:
  Beam 0, Token 0:  is (P=1.0)

Position 7:
  Beam 0, Token 0:  Paris (P=1.0)

Position 8:
  Beam 0, Token 0: . (P=1.0)

Position 9:
  Beam 0, Token 0:  The (P=1.0)

Position 10:
  Beam 0, Token 0:  capital (P=1.0)

Position 11:
  Beam 0, Token 0:  of (P=1.0)

Position 12:
  Beam 0, Token 0:  France (P=1.0)

Position 13:
  Beam 0, Token 0:  is (P=1.0)

Position 14:
  Beam 0, Token 0:  Paris (P=1.0)

Position 15:
  Beam 0, Token 0: . (P=1.0)

Position 16:
  Beam 0, Token 0:  The (P=1.0)

Position 17:
  Beam 0, Token 0:  capital (P=1.0)

Position 18:
  Beam 0, Token 0:  of (P=1.0)

Position 19:
  Beam 0, Token 0:  France (P=1.0)



In [54]:
torch.softmax(output_scores[8], dim=-1).topk(k=5, dim=-1)

torch.return_types.topk(
values=tensor([[0.5000, 0.5000, 0.0000, 0.0000, 0.0000]], device='cuda:0'),
indices=tensor([[1041,   16,    2,    1,    0]], device='cuda:0'))

In [56]:
tokenizer.decode([16])

'1'

In [140]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd
import numpy as np

# Define global variables
n_beams = 3
n_answers_per_question = 3
questions = [
    "What is your favorite colour?",
    "What is the capital of France?",
    "Who wrote 'To Kill a Mockingbird'?"
]
output_texts = []
common_prefixes = []
cumulative_probs = []
adjusted_probs = []
entropy_scores = []
tokens_list = []
token_probs_list = []
prompted_questions = []
full_output_texts = []

def get_prompt(query, strategy):
    if strategy == "Fact Statement":
        return f"{query} Fact:"
    elif strategy == "Completion":
        return f"{query} The answer is:"
    elif strategy == "Definitive Statement":
        return f"The answer to the question '{query}' is:"
    # Add other strategies if needed
    return query

def clean_response(output_text, strategy):
    if strategy == "Fact Statement":
        return output_text.split("Fact:")[-1].strip()
    elif strategy == "Completion":
        return output_text.split("The answer is:")[-1].strip()
    elif strategy == "Definitive Statement":
        return output_text.split("is:")[-1].strip()
    # Add other strategies if needed
    return output_text

def calculate_entropy(probs):
    return -np.sum(probs * np.log(probs))

# Initialize model and tokenizer
model_name = "meta-llama/Meta-Llama-3-8B"
max_new_tokens = 20
temperature = 1
strategy = "Definitive Statement"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
model.eval()

generation_config = {
    "temperature": temperature,
    "do_sample": False,  # Not sampling, to stay consistent with beam search
    "top_p": 0.75,
    "top_k": 40,
    "num_beams": n_beams,
    "num_return_sequences": n_beams,
    "output_scores": True,
    "output_hidden_states": False,
    "output_attentions": False,
    "return_dict_in_generate": True,
    "stop_strings": ["."]
}

for query in questions:
    prompt = get_prompt(query, strategy)
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens, tokenizer=tokenizer)

    for beam_idx in range(n_beams):
        full_output_text = tokenizer.decode(outputs.sequences[beam_idx], skip_special_tokens=True)
        clean_output_text = clean_response(full_output_text, strategy)
        
        # Determine the common prefix
        prefix_length = len(full_output_text) - len(clean_output_text)
        common_prefix = full_output_text[:prefix_length]
        
        # Store the common prefix and the cleaned response
        common_prefixes.append(common_prefix)
        output_texts.append(clean_output_text)
        full_output_texts.append(full_output_text)
        prompted_questions.append(prompt)
        
        # Calculate cumulative probability using sequence scores (log probabilities)
        log_cum_prob = outputs.sequences_scores[beam_idx].item()
        cum_prob = np.exp(log_cum_prob)
        
        # Calculate adjusted probability and entropy score for the generated part
        probs = []
        token_probs = []
        sequence = outputs.sequences[beam_idx]
        
        shift_idx = len(sequence) - len(outputs.scores)  # Position to start processing tokens
        for pos_idx, beam_scores in enumerate(outputs.scores):
            softmax_scores = torch.softmax(beam_scores, dim=-1)
            token_prob = softmax_scores[beam_idx][sequence[pos_idx + shift_idx - 1]].item()
            token_probs.append(round(token_prob, 4))
            probs.append(token_prob)
        
        probs = torch.tensor(probs)
        adj_prob = torch.exp((len(probs) ** -1) * torch.sum(torch.log(probs)))
        
        # Calculate entropy score
        entropy = calculate_entropy(np.array(probs))
        
        # Append results
        cumulative_probs.append(round(cum_prob, 4))
        adjusted_probs.append(round(adj_prob.item(), 4))
        entropy_scores.append(round(entropy, 4))
        tokens_list.append([tokenizer.decode([token]) for token in sequence[shift_idx:]])
        token_probs_list.append(token_probs)

# Create DataFrame
df = pd.DataFrame({
    "Original Question": np.repeat(questions, n_answers_per_question),
    "Prompted Question": prompted_questions,
    "Generated Answer": output_texts,
    "Cumulative Probability": cumulative_probs,
    "Adjusted Probability": adjusted_probs,
    "Entropy Score": entropy_scores,
    "Tokens": tokens_list,
    "Token Probabilities": token_probs_list
})

# Save to Excel
df.to_excel("multinomial_beam_sampling_results.xlsx", index=False)

Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.48s/it]
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.75` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `40` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`. This was detected when initializing the generation config instance, which means the corresponding file may h

In [141]:
torch.exp(outputs.sequences_scores)

tensor([0.6801, 0.6669, 0.6332], device='cuda:0')

In [142]:
df

,Original Question,Prompted Question,Generated Answer,Cumulative Probability,Adjusted Probability,Entropy Score,Tokens,Token Probabilities
0,What is your favorite colour?,The answer to the question 'What is your favor...,'I don't have a favourite colour.',0.3990,0.0255,1.7181,"[I, don, 't, have, a, favourite, colour, ...","[0.2166, 0.0975, 0.0001, 0.0, 0.6982, 0.4581, ..."
1,What is your favorite colour?,The answer to the question 'What is your favor...,'I don't have a favorite colour.',0.3935,0.0252,1.9696,"[I, don, 't, have, a, favorite, colour, ....","[0.2166, 0.1272, 0.4358, 0.9483, 0.0, 0.0, 0.4..."
2,What is your favorite colour?,The answer to the question 'What is your favor...,'I don't have a favourite colour'.,0.3775,0.0001,0.5894,"[I, don, 't, have, a, favourite, colour, ...","[0.2166, 0.0, 0.0, 0.0002, 0.0, 0.0003, 0.0, 0..."
3,What is the capital of France?,The answer to the question 'What is the capita...,Paris.\nThe capital of France is Paris.,0.5398,0.0080,1.5422,"[.\n, The, capital, of, France, is, Paris...","[0.7869, 0.3095, 0.0, 0.3043, 0.0007, 0.0002, ..."
4,What is the capital of France?,The answer to the question 'What is the capita...,Paris.\nThe capital of France is Paris.,0.5250,0.1167,1.8352,"[.\n, The, capital, of, France, is, Paris...","[0.7869, 0.0, 0.2528, 0.3452, 0.8402, 0.9307, ..."
5,What is the capital of France?,The answer to the question 'What is the capita...,Paris. Paris is the capital of France.,0.5122,0.0034,0.8283,"[., Paris, is, the, capital, of, France,...","[0.7869, 0.0, 0.0004, 0.8068, 0.0001, 0.0, 0.0..."
6,Who wrote 'To Kill a Mockingbird'?,The answer to the question 'Who wrote 'To Kill...,"Harper Lee. Harper Lee was born on April 28, 1...",0.6801,0.0000,0.0982,"[ Harper, Lee, ., Harper, Lee, was, born,...","[0.0, 0.0, 0.0, 0.0, 0.0003, 0.011, 0.0002, 0...."
7,Who wrote 'To Kill a Mockingbird'?,The answer to the question 'Who wrote 'To Kill...,"Harper Lee. Harper Lee was born on April 28, 1...",0.6669,0.0000,0.1641,"[ Harper, Lee, ., Harper, Lee, was, born,...","[0.0, 0.0225, 0.0, 0.0003, 0.0, 0.0, 0.0152, 0..."
8,Who wrote 'To Kill a Mockingbird'?,The answer to the question 'Who wrote 'To Kill...,Harper Lee. Harper Lee was born in 1926 in Mon...,0.6332,0.0000,0.1495,"[ Harper, Lee, ., Harper, Lee, was, born,...","[0.0, 0.0, 0.0005, 0.0, 0.0016, 0.0, 0.0002, 0..."
